In [1]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [2]:
# .env 파일에서 환경 변수 로드 (.env 파일에는 OPENAI API 키값을 적으면 됩니다. -> OPENAI_API_KEY=...)
load_dotenv("../..env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = OpenAI()

In [3]:
# <이전 대화를 포함한 메시지 전달>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 금융 상담 역할
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다."),
        ("placeholder", "{messages}"),  # 대화 이력 추가
    ]
)

# 프롬프트와 모델을 연결하여 체인 생성
chain = prompt | chat

# 이전 대화를 포함한 메시지 전달
ai_msg = chain.invoke(
    {
        "messages": [
            ("human", "저축을 늘리기 위해 무엇을 할 수 있나요?"),  # 사용자의 첫 질문
            ("ai", "저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요."),  # 챗봇의 답변
            ("human", "방금 뭐라고 했나요?"),  # 사용자의 재확인 질문
        ],
    }
)
print(ai_msg.content)  # 챗봇의 응답 출력


저축을 늘리기 위해 목표를 설정하고, 매달 자동 이체를 통해 일정 금액을 저축하라는 조언을 드렸습니다. 이렇게 하면 저축을 더 쉽게 관리할 수 있습니다. 추가로, 지출을 점검하고 불필요한 비용을 줄이는 것도 중요합니다.


In [4]:
# <`ChatMessageHistory`를 사용한 메시지 관리>

from langchain_community.chat_message_histories import ChatMessageHistory

# 대화 이력 저장을 위한 클래스 초기화
chat_history = ChatMessageHistory()

# 사용자 메시지 추가
chat_history.add_user_message("저축을 늘리기 위해 무엇을 할 수 있나요?")
chat_history.add_ai_message("저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요.")

# 새로운 질문 추가 후 다시 체인 실행
chat_history.add_user_message("방금 뭐라고 했나요?")
ai_response = chain.invoke({"messages": chat_history.messages})
print(ai_response.content)  # 챗봇은 이전 메시지를 기억하여 답변합니다.

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23624\2792247957.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23624\2792247957.py:6: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  chat_history = ChatMessageHistory()


저축을 늘리기 위해 저축 목표를 설정하고, 매달 자동 이체를 통해 일정 금액을 저축하는 방법을 추천했습니다. 이렇게 하면 저축이 더 쉽게 이루어질 수 있습니다. 추가적인 조언이나 도움이 필요하시면 말씀해 주세요!


In [6]:
# <` RunnableWithMessageHistory`를 사용한 메시지 관리>

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 시스템 메시지와 대화 이력을 사용하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"),  # 이전 대화 이력
        ("human", "{input}"),  # 사용자의 새로운 질문
    ]
)

# 대화 이력을 관리할 체인 설정
chat_history = ChatMessageHistory()
chain = prompt | chat

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,  # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

# 질문 메시지 체인 실행
answer = chain_with_message_history.invoke(
    {"input": "저축을 늘리기 위해 무엇을 할 수 있나요?"},
    {"configurable": {"session_id": "unused"}},
).content

print(answer)

d:\source\langchain-tutorial-main\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


저축을 늘리기 위해 고려할 수 있는 몇 가지 방법은 다음과 같습니다:

1. **예산 세우기**: 수입과 지출을 정리하고, 불필요한 지출을 줄일 수 있는 부분을 찾아보세요. 월별 예산을 세우고 계획에 따라 소비하세요.

2. **지출 기록하기**: 매일 또는 매주 지출을 기록하여 어디에 돈이 많이 쓰이는지 분석하세요. 이를 통해 필요 없는 지출을 줄일 수 있습니다.

3. **자동 이체 설정하기**: 매달 특정 금액을 저축계좌로 자동 이체하도록 설정하여 저축을 습관화하세요. 급여를 받는 즉시 저축하면 사용 가능한 금액이 줄어들어 자연스럽게 저축을 늘릴 수 있습니다.

4. **소비 목표 설정하기**: 저축 목표를 정하고, 이를 달성하기 위해 어떤 금액을 매달 저축할 것인지 계획하세요. 목표가 있으면 저축 동기가 생깁니다.

5. **비상금 마련하기**: 예기치 않은 지출에 대비할 수 있는 비상금을 마련해두면 그만큼 불필요한 긴급 자금을 사용하지 않게 되어 저축을 유지할 수 있습니다.

6. **상여금이나 추가 수입 활용하기**: 보너스, 세금 환급 등 추가 수입이 있을 경우 이를 저축에 할당하세요. 이는 큰 금액을 저축하는 데 도움을 줄 수 있습니다.

7. **저축 상품 활용하기**: 높은 이자율을 제공하는 단기 적금이나 정기예금과 같은 금융 상품을 찾아보세요. 이자 수익을 통해 저축을 더욱 늘릴 수 있습니다.

8. **소규모 소비 습관 변경**: 외식, 카페 음료, 불필요한 쇼핑 등을 줄이고, 필요한 경우에만 지출하세요.

9. **구독 서비스 점검하기**: 사용하지 않는 구독 서비스나 불필요한 회원가입을 정리하여 월 지출을 줄이세요.

10. **자산 재조정**: 현재의 투자 포트폴리오를 점검하고, 수익성이 낮은 자산을 정리하여 더 나은 투자 방향으로 전환할 수 있습니다.

이러한 방법들을 종합적으로 활용하면 저축 수준을 효과적으로 늘릴 수 있습니다. 중요한 것은 지속적인 관리와 자기 통제입니다.


In [7]:
# 새로운 입력 메시지를 추가하고 체인을 실행
answer = chain_with_message_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}  # 세션 ID 설정
).content
print(answer)

당신은 저축을 늘리기 위해 무엇을 할 수 있는지에 대해 질문하셨습니다. 저축을 증가시키기 위한 여러 가지 방법에 대해 설명해 드렸습니다. 궁금한 점이나 더 알고 싶은 내용이 있다면 언제든지 말씀해 주세요!


In [9]:
# <메시지 트리밍 예제>

# 라이브러리 불러오기
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# 메시지 트리밍 유틸리티 설정
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

# 트리밍된 대화 이력과 함께 체인 실행
chain_with_trimming = (
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer)
    | prompt
    | chat
)

# 트리밍된 대화 이력을 사용하는 체인 설정
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming,
    lambda session_id: chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# 새로운 대화 내용 추가 후 체인 실행
answer = chain_with_trimmed_history.invoke(
    {"input": "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}}  # 세션 ID 설정
).content
print(answer)


d:\source\langchain-tutorial-main\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


5년 내에 집을 사오기 위해서 효율적인 재정 계획을 세우는 것은 매우 중요합니다. 아래는 단계별로 도움이 될 수 있는 재정 계획입니다:

### 1단계: 재정 상태 평가
- **수입과 지출 분석**: 자신의 월급, 기타 수입, 고정 지출(임대료, 공과금 등), 변동 지출(식비, 오락비 등)을 분석합니다.
- **순자산 계산**: 자산(현금, 투자, 부동산 등)과 부채(대출, 카드 빚 등)를 계산하여 순자산을 파악합니다.

### 2단계: 주택 구매 예산 설정
- **주택 가격 조사**: 원하는 지역의 평균 주택 가격을 조사합니다.
- **다운페이먼트**: 주택 가격의 10-20%를 다운페이먼트로 준비하는 것을 목표로 합니다. 이를 위해 필요한 금액을 설정합니다.

### 3단계: 저축 계획 수립
- **목표 금액 정하기**: 주택 구매에 필요한 다운페이먼트와 관련 비용을 포함한 총 금액을 설정합니다.
- **저축 방법**: 매달 저축할 금액을 결정하고 별도의 저축 계좌를 개설하여 꾸준히 저축합니다. 이자율이 높은 예금 상품이나 적금도 고려해볼 수 있습니다.
  
### 4단계: 신용 점수 관리
- **신용 점수 확인**: 신용 점수를 정기적으로 확인하고 관리하세요. 신용 점수가 높을수록 대출 금리가 유리해집니다.
- **부채 관리**: 신용 카드 잔액을 낮추고, 제때 결제하는 습관을 기릅니다.

### 5단계: 대출 옵션 조사
- **대출 상품 비교**: 금융 기관의 다양한 주택 대출 상품을 비교하여 가장 적합한 조건을 찾습니다. 고정금리와 변동금리의 장단점을 잘 이해하세요.
- **사전 승인**: 대출 사전 승인을 받으면 구매 과정에서 신뢰성을 높일 수 있습니다.

### 6단계: 추가 비상 자금 마련
- **비상금 조성**: 주택 구매와 함께 예기치 못한 비용에 대비해 최소 3-6개월 분의 생활비를 비상용 계좌에 두도록 합니다.

### 7단계: 시장 조사 및 전문가 상담
- **주택 시장 분석**: 원하는 지역의 부동산 시장 동향을 주의 깊게 모니터링합니

In [10]:
# 새로운 입력 메시지를 추가하고 체인을 실행
answer = chain_with_trimmed_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}}  # 세션 ID 설정
).content
print(answer)

당신은 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하는지에 대해 질문하셨습니다. 제가 재정 계획을 세우는 단계와 방법에 대해 조언을 드렸습니다. 추가적인 질문이나 설명이 필요하시면 말씀해 주세요!


In [11]:
# <이전 대화 요약 내용 기반으로 답변하기>

def summarize_messages(chain_input):
    stored_messages = chat_history.messages
    if len(stored_messages) == 0:
        return False
    # 대화를 요약하기 위한 프롬프트 템플릿 설정
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("placeholder", "{chat_history}"),  # 이전 대화 이력
            (
                "user",
                "이전 대화를 요약해 주세요. 가능한 한 많은 세부 정보를 포함하십시오.",  # 요약 요청 메시지
            ),
        ]
    )

    # 요약 체인 생성 및 실행
    summarization_chain = summarization_prompt | chat
    summary_message = summarization_chain.invoke({"chat_history": stored_messages})
    chat_history.clear()  # 요약 후 이전 대화 삭제
    chat_history.add_message(summary_message)  # 요약된 메시지를 대화 이력에 추가
    return True

In [12]:
# 대화 요약을 처리하는 체인 설정
chain_with_summarization = (
    RunnablePassthrough.assign(messages_summarized=summarize_messages)
    | chain_with_message_history
)

# 요약된 대화를 기반으로 새로운 질문에 응답
print(chain_with_summarization.invoke(
    {"input": "저에게 어떤 재정적 조언을 해주셨나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}  # 세션 ID 설정
).content)


이전 대화에서 드렸던 재정적 조언은 다음과 같습니다:

1. **저축을 늘리기 위한 방법**:
   - **예산 세우기**: 수입과 지출을 파악하고, 불필요한 지출을 줄여 저축할 수 있는 여유를 확보하세요.
   - **지출 기록하기**: 매일의 소비를 기록하여 소비 패턴을 파악하고 필요 없는 지출을 줄이는 데 도움이 됩니다.
   - **자동 이체 설정하기**: 매월 일정 금액을 저축 계좌로 자동 이체하여 저축을 습관화하세요.
   - **소비 목표 설정하기**: 저축 목표를 세우고 그 목표를 달성하기 위한 계획을 세우세요.
   - **비상금 마련하기**: 예기치 못한 상황에 대비하여 일정 금액의 비상금을 확보하세요.
   - **상여금 및 보너스 활용하기**: 추가 소득(상여금, 세금 환급 등)을 저축에 활용하세요.
   - **저축 상품 활용하기**: 높은 이자율을 제공하는 저축 상품이나 투자 상품을 고려하세요.
   - **소규모 소비 습관 변경하기**: 외식이나 불필요한 소비를 줄이는 방법을 생각하세요.
   - **구독 서비스 점검하기**: 사용하지 않는 구독 서비스를 정리하여 지출을 줄이세요.
   - **자산 재조정하기**: 수익성이 낮은 자산을 정리하고 더 나은 투자 옵션으로 전환하세요.

2. **5년 내에 집 구매를 위한 재정 계획**:
   - **재정 상태 평가**: 현재의 수입, 지출, 자산, 부채를 분석하여 전체적인 재정 상태를 파악하세요.
   - **주택 구매 예산 설정**: 원하는 지역의 평균 주택 가격을 조사하고, 다운페이먼트 목표를 설정하세요.
   - **저축 계획 수립**: 매달 저축할 금액을 정하고, 목표 달성을 위한 계획을 세우세요.
   - **신용 점수 관리**: 신용 점수를 점검하고, 필요한 경우 부채를 조정하여 신용을 개선하세요.
   - **대출 옵션 조사**: 다양한 금융 기관의 주택 대출 상품을 비교하고 사전 승인을 받아보세요.
   - **추가 비상 자금 마련**: 3-6개월의 생활비를 비상용으로 준비